# 07 · Executive KPIs & validation

Cierra el proyecto con indicadores de dirección, chequeos de coherencia y reglas de uso responsable.

In [1]:
from pathlib import Path
import sys, json, numpy as np, pandas as pd
ROOT=next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/project8.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Abre el notebook dentro del repositorio del Proyecto 8.')
sys.path.insert(0,str(ROOT))
from src.project8 import *

df=pd.read_csv(ROOT/'data/raw/voyages_2026_synthetic.csv',parse_dates=['date'])
metrics=json.loads((ROOT/'outputs/tables/model_metrics.json').read_text()) if (ROOT/'outputs/tables/model_metrics.json').exists() else fit_xgb(df)[2]
caps=pd.read_csv(ROOT/'outputs/tables/captain_adjusted_comparison.csv') if (ROOT/'outputs/tables/captain_adjusted_comparison.csv').exists() else adjusted_captains(df)[1]
kpis={
'Voyages':len(df),'Fuel t':df.fuel_tonnes.sum(),'CO2e t':df.co2e_tonnes.sum(),'OTR15 %':df.on_time15.mean()*100,'Fuel kg/nm':df.fuel_tonnes.sum()*1000/df.distance_nm.sum(),'CO2e kg/pax':df.co2e_tonnes.sum()*1000/df.passengers.sum(),'Margin €':df.margin_eur.sum(),'XGB WAPE %':metrics['WAPE_pct'],'Adjusted comparisons with support':int(caps.comparison_valid.sum())}
display(pd.DataFrame.from_dict(kpis,orient='index',columns=['value']))

,value
Voyages,7.200000e+03
Fuel t,5.866531e+04
CO2e t,1.836602e+05
OTR15 %,8.359722e+01
Fuel kg/nm,8.188732e+01
CO2e kg/pax,3.599540e+01
Margin €,2.965058e+08
XGB WAPE %,3.367111e+00
Adjusted comparisons with support,1.800000e+01


Checklist final.

In [2]:
checks={
'No duplicated voyages':df.voyage_id.is_unique,
'Margin reconciles':np.allclose(df.margin_eur,df.revenue_eur-df.fuel_cost_eur-df.ets_cost_eur-df.variable_other_eur-df.fixed_trip_eur),
'XGB R2 > .80':metrics['R2']>.80,
'Adjusted captain comparisons require support':bool((caps.loc[caps.comparison_valid,'support_score']>=.60).all())}
print(checks); assert all(checks.values())

{'No duplicated voyages': True, 'Margin reconciles': True, 'XGB R2 > .80': True, 'Adjusted captain comparisons require support': True}
